# 01 EDA

Tablo grain'leri, kullanılabilir tarih aralığı ve aktif seller tanımı. Her tablo için önce grain hipotezi, sonra duplicate testi.

In [183]:
import sys
print(sys.executable)

import seller_propensity
print(seller_propensity.__file__)

/Users/mudogan/Desktop/seller-propensity/.venv/bin/python
/Users/mudogan/Desktop/seller-propensity/src/seller_propensity/__init__.py


In [184]:
import pandas as pd
from pathlib import Path

In [185]:
def short_name(path: Path) -> str:
    """Short name for a dataset file path, removing the prefix and suffix."""
    name = path.stem
    name = name.removeprefix("olist_")
    name = name.removesuffix("_dataset")
    return name

DATA_DIR = Path.cwd().parent / "data" / "raw"
data_dict = {short_name(f): pd.read_csv(f) for f in sorted(DATA_DIR.glob("*.csv"))}

In [186]:
for name, df in data_dict.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
order_items: (112650, 7)
order_payments: (103886, 5)
order_reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
product_category_name_translation: (71, 2)


In [187]:
def count_duplicates(df: pd.DataFrame, keys: str | list[str]) -> int:
    """Count the number of duplicate rows in a DataFrame based on specified keys."""
    return df.duplicated(subset=keys, keep='first').sum()

## 1. orders

In [188]:
data_dict["orders"].head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


**orders için Hipotez:**

1 satır = bir sipariş ve key = order_id

In [189]:
print(f"order_id: {count_duplicates(data_dict['orders'], ['order_id'])}")

order_id: 0


Hipotez doğru.

## 2. order_items

In [190]:
data_dict["order_items"].head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


**order_items için Hipotez:**

İlk tahmin key = order_id. Tutmazsa 1 satır = bir siparişteki bir ürün kalemi ve key = (order_id, order_item_id)

In [191]:
print(f"order_id: {count_duplicates(data_dict['order_items'], ['order_id'])}")
print(f"order_id + order_item_id: {count_duplicates(data_dict['order_items'], ['order_id', 'order_item_id'])}")

order_id: 13984
order_id + order_item_id: 0


order_id tek başına unique değil, bir siparişte birden fazla ürün olabiliyor. order_item_id sipariş içindeki sıra numarası. seller_id de bu tabloda, seller bazlı hesaplar buradan geçecek.

## 3. order_reviews

In [192]:
data_dict["order_reviews"].head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


**order_reviews için Hipotez:**

1 satır = bir review ile bir sipariş eşleşmesi ve key = (order_id, review_id)

In [193]:
print(f"order_id: {count_duplicates(data_dict['order_reviews'], ['order_id'])}")
print(f"review_id: {count_duplicates(data_dict['order_reviews'], ['review_id'])}")
print(f"order_id + review_id: {count_duplicates(data_dict['order_reviews'], ['order_id', 'review_id'])}")

order_id: 551
review_id: 814
order_id + review_id: 0


Bir siparişin birden fazla review'u olabiliyor, bir review da birden fazla siparişe bağlı olabiliyor (N:M). Aynı review'un neden birden fazla siparişte olduğuna 8. bölümde bakıyorum.

## 4. order_payments

In [194]:
data_dict["order_payments"].head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


**order_payments için Hipotez:**

1 satır = bir ödeme kaydı ve key = (order_id, payment_sequential). Bir sipariş birden fazla ödeme yöntemiyle ödenebiliyor.

In [195]:
print(f"order_id: {count_duplicates(data_dict['order_payments'], ['order_id'])}")
print(f"order_id + payment_sequential: {count_duplicates(data_dict['order_payments'], ['order_id', 'payment_sequential'])}")

order_id: 4446
order_id + payment_sequential: 0


Hipotez doğru. Not: sipariş tutarı lazım olursa ödemeleri önce order_id bazında toplamak gerekiyor, yoksa join'de satır çoğalır.

## 5. customers

In [196]:
data_dict["customers"].head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


**customers için Hipotez:**

1 satır = bir müşteri ve key = customer_id

In [197]:
print(f"customer_id: {count_duplicates(data_dict['customers'], ['customer_id'])}")
print(f"customer_unique_id: {count_duplicates(data_dict['customers'], ['customer_unique_id'])}")

customer_id: 0
customer_unique_id: 3345


In [198]:
data_dict["customers"][data_dict["customers"]["customer_unique_id"].duplicated(keep=False)].head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR
13,eabebad39a88bb6f5b52376faec28612,295c05e81917928d76245e842748184d,5704,sao paulo,SP
32,2d5831cb2dff7cdefba62e950ae3dc7b,e9dd12dca17352644a959d9dea133935,42800,camacari,BA
33,b2bed119388167a954382cca36c4777f,e079b18794454de9d2be5c12b4392294,27525,resende,RJ
42,a3b0fda37bae14cf754877bed475e80c,c9158d089637ab443c78984d20da7fc0,5727,sao paulo,SP
88,3f6ede29d4c69cd3316d2035b6cec1fb,7a380cb5434e6b6b5b37d45bb99dbe8a,9890,sao bernardo do campo,SP
113,8891eb5ca0e28df961b2b5b8f3c0eb23,13ac6da47fef1b17b16f511e03b57918,2632,sao paulo,SP
116,fe6d73ac006153a398439253006e5adc,60f874c577dc096ecd52bd0bf6bd5942,36780,astolfo dutra,MG
132,f7c5afab273b47ab517e096e0219b932,b6c083700ca8c135ba9f0f132930d4e8,8686,suzano,SP


In [199]:
data_dict["customers"][data_dict["customers"]["customer_unique_id"] == "4c93744516667ad3b8f1fb645a3116a4"]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC
66803,802bb9a59876a712f8380da8f297057c,4c93744516667ad3b8f1fb645a3116a4,89253,jaragua do sul,SC


Bu örnekte zip kodu farklı ama tek örnekten sonuç çıkmaz. customers ve orders ikisi de 99441 satır, customer_id orders'ta kaç kez geçiyor bakalım.

In [200]:
print(f"orders'ta customer_id: {count_duplicates(data_dict['orders'], ['customer_id'])}")
print(data_dict['customers']['customer_id'].isin(data_dict['orders']['customer_id']).all())

orders'ta customer_id: 0
True


customer_id orders ile 1:1, yani her sipariş için ayrı üretilmiş bir id. Gerçek müşteri customer_unique_id. Grain: 1 satır = bir siparişin müşteri kaydı. Müşteri bazlı hesaplarda customer_unique_id kullanılmalı.

## 6. products

In [201]:
data_dict["products"].head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


**products için Hipotez:**

1 satır = bir ürün ve key = product_id

In [202]:
print(f"product_id: {count_duplicates(data_dict['products'], ['product_id'])}")

product_id: 0


Hipotez doğru.

## 7. sellers

In [203]:
data_dict["sellers"].head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


**sellers için Hipotez:**

1 satır = bir satıcı ve key = seller_id

In [204]:
print(f"seller_id: {count_duplicates(data_dict['sellers'], ['seller_id'])}")

seller_id: 0


Hipotez doğru.

## 8. Birden fazla siparişe bağlı review'lar

Hipotezim: müşteri aynı gün birden fazla sipariş verdi, tek review yazdı ve review hepsine bağlandı. Doğruysa bu siparişler aynı kişiye ve aynı güne ait olmalı.

In [205]:
review_w_order = data_dict["order_reviews"].merge(data_dict["orders"], how='left', on='order_id')
review_w_customer = review_w_order.merge(data_dict["customers"], how='left', on='customer_id')

# join keys are unique on the right side, row count should stay the same
print(len(data_dict["order_reviews"]) == len(review_w_order) == len(review_w_customer))

review_w_customer['order_purchase_date'] = pd.to_datetime(review_w_customer['order_purchase_timestamp']).dt.normalize()

cols = ['order_id', 'customer_id', 'customer_unique_id', 'order_purchase_date', 'order_purchase_timestamp']
shared_reviews = review_w_customer.groupby('review_id')[cols].nunique().query('order_id > 1')

print(f"birden fazla siparişe bağlı review: {len(shared_reviews)}")
for col in cols[1:]:
    print(f"tek {col}: {(shared_reviews[col] == 1).mean():.1%}")

True
birden fazla siparişe bağlı review: 789
tek customer_id: 0.0%
tek customer_unique_id: 100.0%
tek order_purchase_date: 94.8%
tek order_purchase_timestamp: 32.2%


Hepsi aynı customer_unique_id'ye ait, yüzde 95'i aynı gün ama sadece yüzde 32'si aynı saniye. Yani sistem hatası değil, aynı gün verilen siparişlere tek review yazılmış. customer_id ile bakınca yüzde 0 çıkıyor çünkü customer_id her siparişte farklı.

Not: seller bazında review skoru hesaplarken aynı review iki kez sayılmasın.

## 9. Tarihler

Seller aktivitesini hangi tarih kolonuyla ölçeceğime ve verinin hangi aralıkta sağlıklı olduğuna bakıyorum.

In [206]:
review_w_order.dtypes

review_id                          str
order_id                           str
review_score                     int64
review_comment_title               str
review_comment_message             str
review_creation_date               str
review_answer_timestamp            str
customer_id                        str
order_status                       str
order_purchase_timestamp           str
order_approved_at                  str
order_delivered_carrier_date       str
order_delivered_customer_date      str
order_estimated_delivery_date      str
dtype: object

Tarih kolonlarının hepsi str olarak okunmuş. Önce aktivite için hangi kolonu kullanacağıma bakalım.

In [207]:
data_dict["orders"][["order_purchase_timestamp", "order_approved_at"]].isna().sum()

order_purchase_timestamp      0
order_approved_at           160
dtype: int64

order_purchase_timestamp her siparişte dolu, order_approved_at'te ise 160 boş var. Siparişin seller'a ulaştığı an da bu, o yüzden aktiviteyi order_purchase_timestamp üzerinden ölçeceğim.

Adı _timestamp, _date ya da _at ile biten kolonları datetime'a çeviriyorum. Saati atmıyorum, ileride as-of karşılaştırmalarında lazım olacak.

In [208]:
def transform_datetime(df: pd.DataFrame) -> pd.DataFrame:
    df_copy = df.copy()
    
    str_cols = df_copy.select_dtypes(include=['string', 'object']).columns
    
    targets = ('_timestamp', '_date', '_at')
    valid_cols = str_cols[str_cols.str.endswith(targets)]
    
    for col in valid_cols:
        df_copy[col] = pd.to_datetime(df_copy[col])
        
    return df_copy

for name, df in data_dict.items():
    data_dict[name] = transform_datetime(df)

In [209]:
data_dict["orders"].dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [210]:
data_dict["order_items"].dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
dtype: object

In [211]:
print(f"Minimum order purchase timestamp: {data_dict['orders'].order_purchase_timestamp.min()}")
print(f"Maximum order purchase timestamp: {data_dict['orders'].order_purchase_timestamp.max()}")

Minimum order purchase timestamp: 2016-09-04 21:15:19
Maximum order purchase timestamp: 2018-10-17 17:30:18


Min ve max tek başına yeterli değil, aradaki aylarda veri düzenli akıyor mu diye aylık sipariş sayısına bakıyorum. groupby siparişi olmayan ayı hiç göstermez, eksik ay var mı ona da dikkat etmek lazım.

In [212]:
data_dict["orders"].groupby(data_dict["orders"].order_purchase_timestamp.dt.to_period('M')).size()

order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, dtype: int64

2017-01 öncesi çok seyrek ve 2016-11 hiç yok. 2018-09 ve 2018-10'da da neredeyse sipariş yok. 2018-08'in sonunda da bir düşüş olabilir, gün bazında bakıp aynısı başka bir ay sonunda da var mı diye 2018-07 ile karşılaştırıyorum.

In [213]:
data_dict["orders"][(data_dict["orders"].order_purchase_timestamp >= '2018-08-01') & (data_dict["orders"].order_purchase_timestamp < '2018-09-01')].groupby(data_dict["orders"].order_purchase_timestamp.dt.to_period('D')).size()

order_purchase_timestamp
2018-08-01    311
2018-08-02    302
2018-08-03    314
2018-08-04    245
2018-08-05    276
2018-08-06    372
2018-08-07    370
2018-08-08    316
2018-08-09    289
2018-08-10    256
2018-08-11    188
2018-08-12    197
2018-08-13    292
2018-08-14    316
2018-08-15    288
2018-08-16    320
2018-08-17    257
2018-08-18    198
2018-08-19    204
2018-08-20    256
2018-08-21    243
2018-08-22    187
2018-08-23    144
2018-08-24     99
2018-08-25     69
2018-08-26     73
2018-08-27     67
2018-08-28     44
2018-08-29     14
2018-08-30      4
2018-08-31      1
Freq: D, dtype: int64

In [214]:
data_dict["orders"][(data_dict["orders"].order_purchase_timestamp >= '2018-07-01') & (data_dict["orders"].order_purchase_timestamp < '2018-08-01')].groupby(data_dict["orders"].order_purchase_timestamp.dt.to_period('D')).size()

order_purchase_timestamp
2018-07-01    161
2018-07-02    195
2018-07-03    216
2018-07-04    256
2018-07-05    195
2018-07-06    121
2018-07-07     91
2018-07-08    122
2018-07-09    148
2018-07-10    134
2018-07-11    127
2018-07-12    124
2018-07-13    168
2018-07-14    148
2018-07-15    152
2018-07-16    245
2018-07-17    221
2018-07-18    307
2018-07-19    253
2018-07-20    250
2018-07-21    187
2018-07-22    215
2018-07-23    307
2018-07-24    271
2018-07-25    268
2018-07-26    247
2018-07-27    189
2018-07-28    176
2018-07-29    188
2018-07-30    288
2018-07-31    322
Freq: D, dtype: int64

2018-08'de 22'sinden sonra sayılar kademeli olarak düşüyor, 2018-07'nin sonunda böyle bir düşüş yok. Yani bu sezonsallık değil, veri kesintisi. Son sağlam gün 2018-08-22.

2017 boyunca sayıların artması ise büyüme, veri sorunu değil. Boşlukları ve kopuşları kırpıyorum, büyümeyi kırpmıyorum. 2017-01 seyrek ama sürekli, o yüzden dahil.

**Tarih kararları** (N = 30 günlük label window ve en uzun 180 günlük feature window ile):

1. Sağlam veri başı: 2017-01-01
2. Sağlam veri sonu: 2018-08-20 (son sağlam gün 2018-08-22, 2 gün güven payı bıraktım)
3. İlk observation date: 2017-06-30 (veri başı + 180 gün). Daha erken bir tarihte 180 günlük feature'lar eksik geçmişle hesaplanır ve az satan seller ile verisi olmayan seller ayrılamaz.
4. Son observation date: 2018-07-21 (veri sonu - 30 gün). Daha geç bir tarihte label window verinin dışına taşar.

Yani başlangıcı feature window, sonu label window belirliyor. N = 30 alışkanlıktan gelen bir sayı, sıradaki bölümde veriyle bakıyorum.

## 10. N Kaç Olmalı?

Bir seller'ın ardışık iki siparişi arasında kaç gün geçtiğine bakıp N'yi buna göre seçmek istiyorum. Bunun için 1 satırı bir seller'ın bir siparişi olan bir tabloya ihtiyacım var.

In [215]:
DATA_START = '2017-01-01'
DATA_END = '2018-08-20'

filtered_orders = data_dict["orders"][(data_dict["orders"].order_purchase_timestamp >= DATA_START) & (data_dict["orders"].order_purchase_timestamp < pd.Timestamp(DATA_END) + pd.Timedelta(days=1))]
ordered_sellers = filtered_orders.merge(data_dict["order_items"], how='left', on='order_id')[['order_id', 'seller_id', 'order_purchase_timestamp', 'order_status']].sort_values('order_purchase_timestamp')

print("Number of missing values in seller_id: ")
print(ordered_sellers.seller_id.isna().sum())

print("\nValue counts for order_status (where seller_id is missing): ")
print(ordered_sellers[ordered_sellers['seller_id'].isna()]['order_status'].value_counts())

Number of missing values in seller_id: 
717

Value counts for order_status (where seller_id is missing): 
order_status
unavailable    601
canceled       111
created          5
Name: count, dtype: int64


orders'a order_items'ı left join ile bağlayınca 717 satırda seller_id boş geliyor. Bunlar hiç kalemi olmayan siparişler ve statüleri unavailable, canceled ya da created. Yani ya ürün yok, ya sipariş iptal edilmiş, ya da hiç ilerlememiş.

Bu siparişler seller için aktivite sayılmamalı, zaten seller'ları olmadığı için gap hesabına da giremezler. O yüzden inner join'e geçiyorum. Sonra kalemi olan siparişlerde de aktivite sayılmaması gereken bir status var mı ona bakıyorum.

In [216]:
ordered_sellers2 = filtered_orders.merge(data_dict["order_items"], how='inner', on='order_id')[['order_id', 'seller_id', 'order_purchase_timestamp', 'order_status']].sort_values('order_purchase_timestamp')

print("Number of missing values in ordered_sellers2: ")
print(ordered_sellers2.isna().sum().sum())

print("\nValue counts for order_status: ")
print(ordered_sellers2['order_status'].value_counts())

Number of missing values in ordered_sellers2: 
0

Value counts for order_status: 
order_status
delivered     108843
shipped         1171
canceled         526
processing       355
invoiced         340
approved           3
Name: count, dtype: int64


Kalemi olan siparişlerde 526 canceled kalem var. İptal edilen sipariş seller için aktivite değil, bunları da çıkarıyorum. shipped, invoiced, processing ve approved henüz tamamlanmamış ama sipariş seller'a ulaşmış, bunları tutuyorum.

In [217]:
ordered_sellers3 = ordered_sellers2[ordered_sellers2['order_status'] != 'canceled']
print(count_duplicates(ordered_sellers3, keys=['seller_id', 'order_id']))
seller_orders = ordered_sellers3.drop_duplicates()

12414


In [222]:
print(count_duplicates(seller_orders, keys=['seller_id', 'order_id']))
len(seller_orders) == len(ordered_sellers3) - 12414

0


True

Aynı siparişte aynı seller'ın birden fazla kalemi 12414 fazla satır yaratıyordu. Tuttuğum kolonların hepsi sipariş seviyesinde (tarih ve status), bu yüzden tekrarları atmak bilgi kaybettirmiyor. seller_orders'ta 1 satır = bir seller'ın bir siparişi.

Canceled kararından sonra bir soru: order_status hangi tarihteki durumu gösteriyor? Bir sipariş zaman içinde birden fazla status alıyorsa orders'ta aynı order_id birden fazla satırda olmalı.

In [219]:
data_dict['orders'].query("order_status == 'created'")

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
7434,b5359909123fa03c50bdb0cfed07f098,438449d4af8980d107bf04571413a8e7,created,2017-12-05 01:07:52,NaT,NaT,NaT,2018-01-11
9238,dba5062fbda3af4fb6c33b1e040ca38f,964a6df3d9bdf60fe3e7b8bb69ed893a,created,2018-02-09 17:21:04,NaT,NaT,NaT,2018-03-07
21441,7a4df5d8cff4090e541401a20a22bb80,725e9c75605414b21fd8c8d5a1c2f1d6,created,2017-11-25 11:10:33,NaT,NaT,NaT,2017-12-12
55086,35de4050331c6c644cddc86f4f2d0d64,4ee64f4bfc542546f422da0aeb462853,created,2017-12-05 01:07:58,NaT,NaT,NaT,2018-01-08
58958,90ab3e7d52544ec7bc3363c82689965f,7d61b9f4f216052ba664f22e9c504ef1,created,2017-11-06 13:12:34,NaT,NaT,NaT,2017-12-01


In [220]:
data_dict['orders'].order_status.value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [221]:
data_dict['orders'].order_id.duplicated().sum()

np.int64(0)

order_id hiç tekrar etmiyor, her siparişin tek bir status'u var. Yani order_status bir event log değil, verinin çekildiği andaki son durum. "Observation date'te bu sipariş iptal miydi?" sorusunu status ile cevaplayamam, Aşama 2 ve 3'te durumu tarih kolonlarından türetmem gerekecek.

## Özet

| Tablo | 1 satır | Key |
|---|---|---|
| orders | sipariş | order_id |
| order_items | siparişteki ürün kalemi | order_id, order_item_id |
| order_reviews | review ve sipariş eşleşmesi | order_id, review_id |
| order_payments | ödeme kaydı | order_id, payment_sequential |
| customers | siparişin müşteri kaydı | customer_id |
| products | ürün | product_id |
| sellers | satıcı | seller_id |

Tarihler: sağlam veri 2017-01-01 ile 2018-08-20 arası, observation date'ler 2017-06-30 ile 2018-07-21 arası (N = 30 ile).

Aktivite: kalemi olan ve canceled olmayan bir sipariş, tarihi order_purchase_timestamp.